Student ID=126522

Hence 2nd chapter of https://web.stanford.edu/~jurafsky/slp3/ is used.

In [1]:
# !pip install pdf2image
# !pip install pdf2image pillow requests
# !pip install pymupdf

In [2]:
import fitz  # PyMuPDF

pdf_path = "2.pdf"

doc = fitz.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text()



In [3]:
print(len(text))

103958


In [4]:
print(text[:1000])  # preview first 1000 characters

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
2
Words and Tokens
User:
I need some help, that much seems certain.
ELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP
User:
Perhaps I could learn to get along with my mother.
ELIZA: TELL ME MORE ABOUT YOUR FAMILY
User:
My mother takes care of me.
ELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU
User:
My father.
ELIZA: YOUR FATHER
User:
You are like my father in some ways.
Weizenbaum (1966)
The dialogue above is from ELIZA, an early natural language processing system
ELIZA
that could carry on a limited conversation with a user by imitating the responses of
a Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple
program that uses pattern matching on words to recognize phrases like “I need X”
and change the words into suitable outputs like “What would it mean to you if you
got X?”. ELIZA’s mimicry of human conversation, while 

Text cleaning

In [5]:
import re

def clean_text(text):
    
    # remove multiple newlines
    text = re.sub(r"\n+", "\n", text)

    # remove page numbers like "2", "3", etc. on their own line
    text = re.sub(r"\n\d+\n", "\n", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_text = clean_text(text)

print(cleaned_text[:1000])

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER Words and Tokens User: I need some help, that much seems certain. ELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP User: Perhaps I could learn to get along with my mother. ELIZA: TELL ME MORE ABOUT YOUR FAMILY User: My mother takes care of me. ELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU User: My father. ELIZA: YOUR FATHER User: You are like my father in some ways. Weizenbaum (1966) The dialogue above is from ELIZA, an early natural language processing system ELIZA that could carry on a limited conversation with a user by imitating the responses of a Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple program that uses pattern matching on words to recognize phrases like “I need X” and change the words into suitable outputs like “What would it mean to you if you got X?”. ELIZA’s mimicry of human conversation, while ve

Chunking text for RAG / Navie RAG

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


chunks = chunk_text(cleaned_text)

print(len(chunks))
print(chunks[0])

39
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER Words and Tokens User: I need some help, that much seems certain. ELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP User: Perhaps I could learn to get along with my mother. ELIZA: TELL ME MORE ABOUT YOUR FAMILY User: My mother takes care of me. ELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU User: My father. ELIZA: YOUR FATHER User: You are like my father in some ways. Weizenbaum (1966) The dialogue above is from ELIZA, an early natural language processing system ELIZA that could carry on a limited conversation with a user by imitating the responses of a Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple program that uses pattern matching on words to recognize phrases like “I need X” and change the words into suitable outputs like “What would it mean to you if you got X?”. ELIZA’s mimicry of human conversation, while

Add metadata

In [7]:
docs = []

for i, chunk in enumerate(chunks):
    docs.append({
        "id": i,
        "text": chunk,
        "source": "Jurafsky NLP Book Chapter 2"
    })

In [ ]:
# 1) QA pair generation (at least 20) based on chapter content.
qa_pairs = [
    {"question": "What is the focus of Chapter 2 in Jurafsky & Martin?", "answer": "It introduces n-gram language models and evaluates their probability estimations and smoothing."},
    {"question": "Define a bigram model in language modeling.", "answer": "A bigram model predicts each word conditioned only on the previous word, P(w_i|w_{i-1})."},
    {"question": "What problem does data sparsity create for n-grams?", "answer": "Many n-grams never occur in training data, giving zero probability to valid sentences."},
    {"question": "What is maximum likelihood estimation for n-gram probabilities?", "answer": "Count-based ratio: P(w_i|context) = count(context,w_i)/count(context)."},
    {"question": "Explain Add-1 (Laplace) smoothing.", "answer": "It adds one to all n-gram counts and normalizes so no event has zero probability."},
    {"question": "What is linear interpolation?", "answer": "Combining unigram, bigram, trigram probabilities with weights to smooth estimates."},
    {"question": "What is backoff in smoothing?", "answer": "Use higher-order model when available, else back off to lower-order distributions."},
    {"question": "What are perplexity and cross-entropy?", "answer": "Perplexity measures model uncertainty; cross-entropy is average log-loss (exponential relation)."},
    {"question": "How does the chapter describe the chain rule for language models?", "answer": "It decomposes sentence probability to product of conditional word probabilities."},
    {"question": "What is the purpose of held-out data in smoothing?", "answer": "To tune interpolation weights or discount parameters without overfitting training data."},
    {"question": "What is a language model's n-gram order effect?", "answer": "Higher order captures more context but requires more data and can overfit without smoothing."},
    {"question": "What is the Katz backoff formula?", "answer": "Discount observed counts and redistribute probability mass to lower-order n-grams via backoff weights."},
    {"question": "Why do we normalize probabilities in smoothing?", "answer": "To ensure all next-word probabilities sum to one per context."},
    {"question": "What is deleted interpolation?", "answer": "Estimating interpolation weights by optimizing on held-out data with data partitions."},
    {"question": "What is the main drawback of Add-1 smoothing for large vocabularies?", "answer": "It over-inflates rare events and distorts probabilities because adding one is too large."},
    {"question": "How are smoothing methods evaluated?", "answer": "By computing held-out perplexity or cross-entropy on test data."},
    {"question": "What tradeoff does smoothing control?", "answer": "Bias vs variance in n-gram probability estimates."},
    {"question": "Why are unigram counts falling short for language modeling?", "answer": "They ignore context, so cannot model word dependencies."},
    {"question": "What is absolute discounting concept?", "answer": "Subtract a constant from non-zero n-gram counts and allocate mass to lower-order distributions."},
    {"question": "How does chapter 2 categorize language modeling tasks?", "answer": "As predicting next word probabilities and scoring sentences for generation or recognition."},
]

In [ ]:
# 2) Embed chunks using sentence-transformers and build retrievers.
!pip install -q sentence-transformers transformers rouge-score


In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_vectors = embed_model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

# Naive retriever
nn_naive = NearestNeighbors(n_neighbors=4, metric='cosine').fit(chunk_vectors)

def retrieve_chunks(query, top_k=3):
    qv = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    dist, idx = nn_naive.kneighbors(qv, n_neighbors=top_k)
    # convert cosine distance to similarity
    return [(i, 1 - float(d)) for i, d in zip(idx[0], dist[0])]

# 3) Generator model (Flan-T5 small) for answer generation
from transformers import pipeline

generator = pipeline('text2text-generation', model='google/flan-t5-small', device=-1)

# ensure deterministic outputs for evaluation

def generate_answer(question, selected_chunks):
    context_text = "\n\n".join([f"Chunk {i}: {chunks[i]}" for i, _ in selected_chunks])
    prompt = f"Given context from a chapter, answer this question.\n\nContext:\n{context_text}\n\nQuestion: {question}\nAnswer:"
    out = generator(prompt, max_length=200, do_sample=False)
    return out[0]['generated_text'].strip()

# 4) Contextual enrichment of chunks (simple heuristic for no external API)

def enrich_chunk(chunk, document_text, title="Jurafsky NLP Book Chapter 2"):
    # choose beginning snippet as context sentence
    prefix = chunk.split('.')[:1][0].strip()
    if len(prefix) > 0:
        return f"This chunk from {title} discusses: {prefix}.\n\n{chunk}"
    return chunk

contextual_chunks = [enrich_chunk(c, cleaned_text) for c in chunks]
contextual_vectors = embed_model.encode(contextual_chunks, convert_to_numpy=True, normalize_embeddings=True)
nn_contextual = NearestNeighbors(n_neighbors=4, metric='cosine').fit(contextual_vectors)

def retrieve_contextual(query, top_k=3):
    qv = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    dist, idx = nn_contextual.kneighbors(qv, n_neighbors=top_k)
    return [(i, 1 - float(d)) for i, d in zip(idx[0], dist[0])]

# 5) Run 20 QA pairs through both pipelines
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

eval_rows = []
for pair in qa_pairs:
    q = pair['question']
    g = pair['answer']

    naive_hits = retrieve_chunks(q, top_k=3)
    naive_ans = generate_answer(q, naive_hits)

    context_hits = retrieve_contextual(q, top_k=3)
    # use contextual text in generator prompt
    context_text = "\n\n".join([f"Chunk {i}: {contextual_chunks[i]}" for i, _ in context_hits])
    context_prompt = f"Given contextual augmented chunks from chapter 2, answer question.\n\nContext:\n{context_text}\n\nQuestion: {q}\nAnswer:"
    contextual_ans = generator(context_prompt, max_length=200, do_sample=False)[0]['generated_text'].strip()

    scores_naive = scorer.score(g, naive_ans)
    scores_context = scorer.score(g, contextual_ans)

    eval_rows.append({
        'question': q,
        'ground_truth_answer': g,
        'naive_rag_answer': naive_ans,
        'contextual_retrieval_answer': contextual_ans,
        'naive_rouge1': scores_naive['rouge1'].fmeasure,
        'naive_rouge2': scores_naive['rouge2'].fmeasure,
        'naive_rougeL': scores_naive['rougeL'].fmeasure,
        'context_rouge1': scores_context['rouge1'].fmeasure,
        'context_rouge2': scores_context['rouge2'].fmeasure,
        'context_rougeL': scores_context['rougeL'].fmeasure,
    })

# 6) Compute average ROUGE and display
import pandas as pd
results_df = pd.DataFrame(eval_rows)
summary = {
    'Method': ['Naive RAG', 'Contextual Retrieval'],
    'ROUGE-1': [results_df['naive_rouge1'].mean(), results_df['context_rouge1'].mean()],
    'ROUGE-2': [results_df['naive_rouge2'].mean(), results_df['context_rouge2'].mean()],
    'ROUGE-L': [results_df['naive_rougeL'].mean(), results_df['context_rougeL'].mean()],
}
summary_df = pd.DataFrame(summary)
print(summary_df)

# 7) Write JSON evaluation file in answer folder
import os, json
os.makedirs('answer', exist_ok=True)
output_file = 'answer/response-st126522-chapter-2.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump([{
        'question': r['question'],
        'ground_truth_answer': r['ground_truth_answer'],
        'naive_rag_answer': r['naive_rag_answer'],
        'contextual_retrieval_answer': r['contextual_retrieval_answer']
    } for r in eval_rows], f, ensure_ascii=False, indent=2)

print(f"JSON evaluation output saved to {output_file}")

# 8) Optionally show first 3 results
results_df[['question','naive_rag_answer','contextual_retrieval_answer']].head(3)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (2330 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. 

                 Method   ROUGE-1   ROUGE-2   ROUGE-L
0             Naive RAG  0.028163  0.001026  0.028163
1  Contextual Retrieval  0.030170  0.000000  0.027563
JSON evaluation output saved to answer/response-st126522-chapter-2.json


,question,naive_rag_answer,contextual_retrieval_answer
0,What is the focus of Chapter 2 in Jurafsky & M...,Words grow without end leads to a problem for ...,"a vocabulary with 7 tokens A, B, C, D, E, AB, ..."
1,Define a bigram model in language modeling.,byte- pair encoding,Byte pair encoding is suboptimal for language ...
2,What problem does data sparsity create for n-g...,unanswerable,unanswerable
